<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/4_Dataset_Recovery_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q yfinance transformers deep-translator feedparser pandas numpy torch

import pandas as pd
import numpy as np
import feedparser
import yfinance as yf
import torch
import urllib.parse
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline
from deep_translator import GoogleTranslator


finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0 if torch.cuda.is_available() else -1)
translator = GoogleTranslator(source='id', target='en')

master_df = pd.read_csv('BBCA_Master_Dataset_BiLSTM.csv')
master_df['Date'] = pd.to_datetime(master_df['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')

In [ ]:
search_keyword = "saham BBCA"
search_range = "14d"

print(f"Initiating Time Machine: Scraping Google News for the past {search_range}...")
encoded_search_keyword = urllib.parse.quote_plus(search_keyword)
rss_url = f"https://news.google.com/rss/search?q={encoded_search_keyword}+when:{search_range}&hl=id&gl=ID&ceid=ID:id"
feed = feedparser.parse(rss_url)

recovery_news = []

for entry in feed.entries:
    # Parse the publication date to YYYY-MM-DD
    pub_date = pd.to_datetime(entry.published).strftime('%Y-%m-%d')
    headline = entry.title

    try:
        english_text = translator.translate(headline)
        label = finbert(english_text)[0]['label']

        if label == 'positive': score = 1
        elif label == 'negative': score = -1
        else: score = 0
    except:
        score = 0

    recovery_news.append({'Date': pub_date, 'Sentiment_Score': score})

if not recovery_news:
    print(" No news found in the last 14 days.")
    df_recovery_sentiment = pd.DataFrame(columns=['Date', 'Sentiment_Score'])
else:
    # Convert to DataFrame and calculate the daily average for the missing days
    df_recovery_news = pd.DataFrame(recovery_news)
    df_recovery_sentiment = df_recovery_news.groupby('Date')['Sentiment_Score'].mean().reset_index()
    print(f" Successfully recovered and scored news for {len(df_recovery_sentiment)} different days.")
    display(df_recovery_sentiment.head())

In [ ]:
bbca_data = yf.download("BBCA.JK", period="20d", progress=False)
if isinstance(bbca_data.columns, pd.MultiIndex):
    bbca_data.columns = bbca_data.columns.droplevel(1)

# Reset index so 'Date' becomes a normal column
bbca_data = bbca_data.reset_index()
bbca_data['Date'] = bbca_data['Date'].dt.strftime('%Y-%m-%d')

# Filter exactly the columns we need
recent_stocks = bbca_data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

print(f"Downloaded {len(recent_stocks)} recent trading days.")

In [ ]:
print("Fusing recovered data...")

# Left join: Attach the recovered daily sentiment to the trading days
recovery_block = pd.merge(recent_stocks, df_recovery_sentiment, on='Date', how='left')
recovery_block['Sentiment_Score'] = recovery_block['Sentiment_Score'].fillna(0)

print("Patching the Master Database...")

# 1. Remove any rows in the Master CSV that have dates overlapping with our new recovery block
master_df = master_df[~master_df['Date'].isin(recovery_block['Date'])]

# 2. Append the fresh recovery block to the bottom
master_df = pd.concat([master_df, recovery_block], ignore_index=True)

# 3. Sort chronologically just to be absolutely safe
master_df = master_df.sort_values('Date').reset_index(drop=True)

# Save the healed database
master_df.to_csv('BBCA_Master_Dataset_LSTM.csv', index=False)

print("="*50)
print("The AI's short-term memory has been successfully restored.")
print("You may now run Notebook 3 (Live Analyst) to get tomorrow's prediction.")
print("="*50)
display(master_df.tail(5))